# PredictIQ — Full ML Training Pipeline

End-to-end XGBoost churn prediction: EDA → Feature Engineering → Training → Evaluation → SHAP Explainability.

In [ ]:
# !pip install scikit-learn xgboost shap pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve, confusion_matrix,
                              ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

plt.style.use('dark_background')
print("Libraries loaded ✓")

## 1. Generate / Load Data

In [ ]:
np.random.seed(42)
n = 10_000

contract_map = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}

age             = np.random.randint(18, 80, n)
tenure          = np.random.randint(0, 120, n)
monthly_charges = np.random.uniform(20, 150, n).round(2)
support_calls   = np.random.poisson(2, n).clip(0, 15)
contract_enc    = np.random.choice([0, 1, 2], n, p=[0.55, 0.25, 0.20])
satisfaction    = np.random.randint(1, 6, n)

log_odds = (
    -3.0
    + 0.4  * (support_calls / 10)
    - 0.6  * (tenure / 120)
    - 0.5  * ((satisfaction - 1) / 4)
    + 0.6  * (contract_enc == 0).astype(float)
    - 0.4  * (contract_enc == 2).astype(float)
    + 0.3  * (monthly_charges > 100).astype(float)
    + 0.02 * np.random.randn(n)
)
prob  = 1 / (1 + np.exp(-log_odds))
churn = (np.random.rand(n) < prob).astype(int)

FEATURES = ['age', 'tenure', 'monthly_charges', 'support_calls', 'contract_encoded', 'satisfaction']
df = pd.DataFrame({
    'age': age, 'tenure': tenure, 'monthly_charges': monthly_charges,
    'support_calls': support_calls, 'contract_encoded': contract_enc,
    'satisfaction': satisfaction, 'churn': churn
})

print(f"Dataset: {len(df):,} rows  |  Churn rate: {churn.mean():.1%}")
df.head()

## 2. Train / Test Split & Model Training

In [ ]:
X = df[FEATURES]
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    use_label_encoder=False, eval_metric='logloss', random_state=42
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print("Training complete ✓")

## 3. Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
cv     = cross_val_score(model, X, y, cv=StratifiedKFold(5), scoring='roc_auc')

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1:        {f1_score(y_test, y_pred):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob):.4f}")
print(f"CV AUC:    {cv.mean():.4f} ± {cv.std():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[0].plot(fpr, tpr, color='#34d399', lw=2, label=f'AUC = {roc_auc_score(y_test, y_prob):.3f}')
axes[0].plot([0,1],[0,1], '--', color='#374151')
axes[0].set_xlabel('False Positive Rate', color='#94a3b8')
axes[0].set_ylabel('True Positive Rate', color='#94a3b8')
axes[0].set_title('ROC Curve', color='white')
axes[0].legend()

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix', color='white')

plt.tight_layout()
plt.savefig('../diagrams/model_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. SHAP Explainability

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Global feature importance
fig, ax = plt.subplots(figsize=(8, 4))
mean_shap = np.abs(shap_values).mean(axis=0)
feat_names_display = ['Customer Age', 'Tenure (months)', 'Monthly Charges',
                       'Support Calls', 'Contract Type', 'Satisfaction']
sorted_idx = np.argsort(mean_shap)
colors = ['#6366f1' if i == sorted_idx[-1] else '#3b82f6' for i in range(len(mean_shap))]
ax.barh([feat_names_display[i] for i in sorted_idx],
        mean_shap[sorted_idx], color=[colors[i] for i in sorted_idx])
ax.set_xlabel('Mean |SHAP value|', color='#94a3b8')
ax.set_title('Global Feature Importance (SHAP)', color='white')
ax.tick_params(colors='#94a3b8')
for spine in ax.spines.values(): spine.set_color('#374151')
plt.tight_layout()
plt.savefig('../diagrams/shap_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nTop feature:", feat_names_display[np.argmax(mean_shap)])